In [24]:
import plotly.graph_objects as go

def twh_to_ej(twh):
    """
    Convert Terawatt-hours (TWh) to Exajoules (EJ).

    Parameters:
    twh (float): Energy in Terawatt-hours.

    Returns:
    float: Energy in Exajoules.
    """
    conversion_factor = 0.0036
    return twh * conversion_factor

def xy(d):
    return zip(*sorted(d.items())) if d else ([], [])

# Hydro

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

* Conventional Hydro: Korea have generated and is planning to generate 3.7 TWh / year. We assume the conventional hydro generation is remained in the future.
* Pumped Hydro: included in `Others (기타)` of `Projection of Generation Mix (발전량 전망)`(p.51, BPESD). As of 2023, pumped hydro generated 3.8 TWh (ES) and `Others` generated 8.3 thus `Pumped Hydro` takes $\frac{3.8}{8.3}=45\%$  of `Others`. We assume this share is maintained in the future.

* Projected Generation of `Others` (TWh) (p.51, BPESD):
    * 2025: 9.3
    * 2030: 11.8
    * 2035: 28.5

In [17]:
print(f"year = 2025, pumped hydro (TWh) = {9.3 * (3.8/8.3):.1f}")
print(f"year = 2030, pumped hydro (TWh) = {11.8 * (3.8 / 8.3):.1f}")
print(f"year = 2035, pumped hydro (TWh) = {28.5 * (3.8 / 8.3):.1f}")

year = 2025, pumped hydro (TWh) = 4.3
year = 2030, pumped hydro (TWh) = 5.4
year = 2035, pumped hydro (TWh) = 13.0


* Predicted Generation of Pumped Hydro:
    * 2025: 4.3
    * 2030: 5.4
    * 2035: 13.0

* Total Hydro (TWh):
    * 2025: 8.0
    * 2030: 9.1
    * 2035: 16.7

In [20]:
dictCapTWh = {2020: 7.15, 2025: 8.0, 2030: 9.1, 2035: 16.7}

In [23]:
print(f"year = 2020, total hydro (EJ) = {twh_to_ej(dictCapTWh[2020]):.4f}")
print(f"year = 2025, total hydro (EJ) = {twh_to_ej(dictCapTWh[2025]):.4f}")
print(f"year = 2030, total hydro (EJ) = {twh_to_ej(dictCapTWh[2030]):.4f}")
print(f"year = 2035, total hydro (EJ) = {twh_to_ej(dictCapTWh[2035]):.4f}")

year = 2020, total hydro (EJ) = 0.0257
year = 2025, total hydro (EJ) = 0.0288
year = 2030, total hydro (EJ) = 0.0328
year = 2035, total hydro (EJ) = 0.0601


Implemented input File: `/input/policy/korea-2035/power/hydro-fixedOutput.xml`

```xml
<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <supplysector name="electricity">
        <subsector name="hydro">
          <stub-technology name="hydro">
            <period year="2020">
              <fixedOutput>0.0257</fixedOutput>
            </period>
            <period year="2025">
              <fixedOutput>0.0288</fixedOutput>
            </period>
            <period year="2030">
              <fixedOutput>0.0328</fixedOutput>
            </period>
            <period year="2035">
              <fixedOutput>0.0601</fixedOutput>
            </period>
          </stub-technology>
        </subsector>
      </supplysector>
    </region>
  </world>
</scenario>

```

In [31]:
years_cap, values_cap = xy(dictCapTWh)

fig = go.Figure()

for name, x, y, dash in [
    ("Current Policies", years_cap, values_cap, None),
]:
    fig.add_trace(go.Scatter(
        x=list(x), y=list(y),
        mode='lines+markers',
        name=name,
        line=(dict(dash=dash) if dash else None)
    ))

# Build annotations without repeating blocks
target_years = [2020, 2025, 2030, 2035]
annotations = []
for yr in target_years:
    val = dictCapTWh.get(yr)
    if val is not None:
        annotations.append(go.layout.Annotation(
            x=yr, y=val,
            xanchor='center', yanchor='bottom',
            text=f"{val:.1f} TWh",
            showarrow=True, arrowhead=1, ax=0, ay=-20
        ))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=800, height=600,
    annotations=annotations,
    xaxis=dict(
        title='Year',
        title_font=dict(size=18),
        tickfont=dict(size=15),
        tickvals=[2020, 2025, 2030, 2035],  # custom tick positions
        range=[2018, 2036]                  # xrange
    ),
    yaxis=dict(
        title='TWh',
        title_font=dict(size=18),
        tickfont=dict(size=15),
        range=[0, 25]                       # yrange
    ),
)

fig.write_image("../figures/hydro_generation.png", scale=2)
fig.show()